In [1]:
import pandas as pd
import numpy as np

# =============================================================================
# DATA LOADING
# Note: Now using parquet files (Broadridge_Core and Broadridge_Activity)
# instead of the original CSV files. These contain the same data but are
# pre-joined and more complete.
# =============================================================================

core = pd.read_parquet('Broadridge_Core_All.parquet')
activity = pd.read_parquet('Broadridge_Activity_All.parquet')

# Force numeric on all money columns
money_cols = [
    'CYTDTransfer', 'PYTDTransfer',
    'CYTDDirectRollover', 'PYTDDirectRollover',
    'CYTDIRAContribution', 'PYTDIRAContribution'
]
for col in money_cols:
    activity[col] = pd.to_numeric(activity[col], errors='coerce')

activity['DeltaDate'] = pd.to_datetime(activity['DeltaDate'], errors='coerce')
activity = activity.sort_values(['FinancialAccountId', 'DeltaDate'])

core['PartyAge'] = pd.to_numeric(core['PartyAge'], errors='coerce')

In [2]:
# =============================================================================
# Step 1: Family Role Distribution
# Source: RelationToAccountOwner field in Broadridge_Core_All.parquet
# =============================================================================

print("=" * 60)
print("[1] RelationToAccountOwner Distribution")
print("=" * 60)
print(core['RelationToAccountOwner'].value_counts(dropna=False))

[1] RelationToAccountOwner Distribution
RelationToAccountOwner
NULL                           129339
Domestic Partner/Non-spouse     70195
Spouse                          53497
Child                           10370
Other relative                   6826
Parent                           4721
Trust                            3673
Sibling                          2806
Estate/Entity                    1782
No relation                      1322
NON-SPOUSE                          4
OTHER RELATIVE                      3
Name: count, dtype: int64


In [3]:
# =============================================================================
# Step 2: Isolate Parent-Child Clusters
# Logic: identify accounts where Child or Parent is listed as beneficiary.
# Spouse and Sibling are excluded — they represent same-generation transfers
# and fall outside the scope of intergenerational wealth transfer.
# =============================================================================

child_accounts  = core[core['RelationToAccountOwner'] == 'Child'][['FinancialAccountId', 'bar_PartyId']].rename(columns={'bar_PartyId': 'ChildPartyId'})
parent_accounts = core[core['RelationToAccountOwner'] == 'Parent'][['FinancialAccountId', 'bar_PartyId']].rename(columns={'bar_PartyId': 'ParentPartyId'})

child_acct_ids  = set(child_accounts['FinancialAccountId'])
parent_acct_ids = set(parent_accounts['FinancialAccountId'])
both_acct_ids   = child_acct_ids & parent_acct_ids
family_acct_ids = child_acct_ids | parent_acct_ids

total_accounts  = core['FinancialAccountId'].nunique()

print("\n" + "=" * 60)
print("[2] Family Cluster Summary")
print("=" * 60)
print(f"  Total unique accounts              : {total_accounts:,}")
print(f"  Accounts with Child beneficiary    : {len(child_acct_ids):,}")
print(f"  Accounts with Parent beneficiary   : {len(parent_acct_ids):,}")
print(f"  Accounts with BOTH (strong signal) : {len(both_acct_ids):,}")
print(f"  Total family-linked accounts       : {len(family_acct_ids):,}")
print(f"  As % of all unique accounts        : {len(family_acct_ids) / total_accounts * 100:.1f}%")


[2] Family Cluster Summary
  Total unique accounts              : 83,535
  Accounts with Child beneficiary    : 10,253
  Accounts with Parent beneficiary   : 4,162
  Accounts with BOTH (strong signal) : 231
  Total family-linked accounts       : 14,184
  As % of all unique accounts        : 17.0%


In [4]:
# =============================================================================
# Step 3: Children per Account
# Shows whether wealth is concentrated in one heir or split across multiple.
# =============================================================================

children_per_account = (
    child_accounts
    .groupby('FinancialAccountId')['ChildPartyId']
    .count()
    .reset_index()
    .rename(columns={'ChildPartyId': 'NumChildren'})
)

single_child_pct = (children_per_account['NumChildren'] == 1).mean() * 100

print("=" * 60)
print("[3] Children per Account")
print("=" * 60)
print(children_per_account['NumChildren'].value_counts().rename("Accounts").to_string())
print(f"\n  Accounts naming only 1 child: {single_child_pct:.1f}%")
print(f"  Wealth is concentrated in a single heir in the vast majority of cases.")

[3] Children per Account
NumChildren
1    10136
2      117

  Accounts naming only 1 child: 98.9%
  Wealth is concentrated in a single heir in the vast majority of cases.


In [5]:
# =============================================================================
# Step 4: Transfer Activity — Family vs Non-Family (Mean Comparison)
#
# Data source: Internal_Wide_All.parquet
# This file contains all accounts with their role and activity data pre-joined,
# giving a more complete population for mean comparison.
#
# Note: Mean comparison using mean() is NOT affected by the duplicate-row
# (CYTD cumulative) issue. That issue only affects total/sum calculations.
# Mean across rows is a valid comparison method here.
# =============================================================================

internal = pd.read_parquet('Internal_Wide_All.parquet',
                            columns=['FinancialAccountId', 'RelationToAccountOwner',
                                     'CYTDTransfer', 'CYTDDirectRollover', 'CYTDIRAContribution',
                                     'PYTDTransfer', 'PYTDDirectRollover', 'PYTDIRAContribution'])

mean_cols = ['CYTDTransfer', 'PYTDTransfer',
             'CYTDDirectRollover', 'PYTDDirectRollover',
             'CYTDIRAContribution', 'PYTDIRAContribution']

for col in mean_cols:
    internal[col] = pd.to_numeric(internal[col], errors='coerce')

# Use the same family account IDs defined in Step 2
fam_iw  = internal[internal['FinancialAccountId'].isin(family_acct_ids)]
nfam_iw = internal[~internal['FinancialAccountId'].isin(family_acct_ids)]

print("=" * 60)
print("[4] Mean Transfer Amounts — Family vs Non-Family")
print("    Source: Internal_Wide_All.parquet")
print("=" * 60)
print(f"  {'Column':<30} {'Family ($)':>15} {'Non-Family ($)':>15}")
for col in mean_cols:
    fm = fam_iw[col].mean()
    nf = nfam_iw[col].mean()
    print(f"  {col:<30} {fm:>15,.2f} {nf:>15,.2f}")

rollover_ratio = nfam_iw['CYTDDirectRollover'].mean() / fam_iw['CYTDDirectRollover'].mean()
print(f"\n  Non-family rollover is {rollover_ratio:.1f}x higher than family rollover.")
print(f"  Interpretation: family beneficiaries receive wealth passively via Transfer;")
print(f"  they do not actively move money themselves.")
print(f"\n  Slide numbers (verified):")
print(f"    Avg CYTD Transfer  — Family: ${fam_iw['CYTDTransfer'].mean():,.0f}  |  Non-Family: ${nfam_iw['CYTDTransfer'].mean():,.0f}")
print(f"    Avg Direct Rollover— Family: ${fam_iw['CYTDDirectRollover'].mean():,.0f}  |  Non-Family: ${nfam_iw['CYTDDirectRollover'].mean():,.0f}")

[4] Mean Transfer Amounts — Family vs Non-Family
    Source: Internal_Wide_All.parquet
  Column                              Family ($)  Non-Family ($)
  CYTDTransfer                         41,554.96       30,528.03
  PYTDTransfer                         34,348.48       22,425.39
  CYTDDirectRollover                   14,664.59       43,628.56
  PYTDDirectRollover                   29,903.28       39,237.67
  CYTDIRAContribution                   2,007.20        1,745.46
  PYTDIRAContribution                   2,811.74        2,567.42

  Non-family rollover is 3.0x higher than family rollover.
  Interpretation: family beneficiaries receive wealth passively via Transfer;
  they do not actively move money themselves.

  Slide numbers (verified):
    Avg CYTD Transfer  — Family: $41,555  |  Non-Family: $30,528
    Avg Direct Rollover— Family: $14,665  |  Non-Family: $43,629


In [7]:
# =============================================================================
# Step 5: Temporal Pattern by Year — DELTA CHANGE METHOD
#
# IMPORTANT FIX from v1.0:
# CYTDTransfer is a cumulative snapshot value. The same account can appear
# multiple times in a year (one row per month). Summing all rows directly
# leads to double-counting.
#
# Correct approach (Delta Change):
#   For each account, take only the LAST record per year.
#   This represents the true year-end cumulative total for that account.
#   Then sum across accounts to get the actual annual total.
# =============================================================================

family_activity = activity[activity['FinancialAccountId'].isin(family_acct_ids)].copy()
family_activity['Year'] = family_activity['DeltaDate'].dt.year

# Take the last snapshot per account per year
last_per_account_year = (
    family_activity
    .sort_values('DeltaDate')
    .groupby(['FinancialAccountId', 'Year'])
    .last()
    .reset_index()
)

yearly = (
    last_per_account_year
    .groupby('Year')
    .agg(
        NumAccounts         = ('FinancialAccountId', 'count'),
        NumWithTransfer     = ('CYTDTransfer', lambda x: (x > 0).sum()),
        TotalTransfer       = ('CYTDTransfer', 'sum'),
        MeanTransfer_all    = ('CYTDTransfer', 'mean'),
        MeanTransfer_active = ('CYTDTransfer', lambda x: x[x > 0].mean()),
        TotalDirectRollover = ('CYTDDirectRollover', 'sum'),
    )
    .reset_index()
)

print("\n" + "=" * 60)
print("[5] Temporal Transfer Pattern by Year (Delta Change Method)")
print("=" * 60)
print(yearly.to_string(index=False))
print("\n  Note: TotalTransfer uses last snapshot per account per year to avoid")
print("  double-counting cumulative CYTD values.")


[5] Temporal Transfer Pattern by Year (Delta Change Method)
 Year  NumAccounts  NumWithTransfer  TotalTransfer  MeanTransfer_all  MeanTransfer_active  TotalDirectRollover
 2024         4592                0           0.00               NaN                  NaN                 0.00
 2025         3093              152    29039846.97      38565.533825        191051.624803          23693594.79
 2026         2883               23     5229242.66      19296.098376        227358.376522            234491.15

  Note: TotalTransfer uses last snapshot per account per year to avoid
  double-counting cumulative CYTD values.


In [8]:
# =============================================================================
# Step 6: Parent Account Holder Age — Transfer Urgency
# Source: Primary Account Owner records in Broadridge_Core for family accounts.
# Shows how near-term the transfer timeline is.
# =============================================================================

family_primary_owners = core[
    (core['FinancialAccountId'].isin(child_acct_ids)) &
    (core['bar_FinancialAccountRole'] == 'Primary Account Owner')
]

parent_ages = family_primary_owners['PartyAge'].dropna()

print("\n" + "=" * 60)
print("[6] Parent Account Holder Age Distribution")
print("=" * 60)
print(f"  Mean age              : {parent_ages.mean():.1f}")
print(f"  Median age            : {parent_ages.median():.1f}")
print(f"  Over 60               : {(parent_ages >= 60).mean() * 100:.0f}%")
print(f"  Over 70               : {(parent_ages >= 70).mean() * 100:.0f}%")
print(f"  Over 80               : {(parent_ages >= 80).mean() * 100:.0f}%")

age_bins = pd.cut(
    parent_ages,
    bins=[0, 50, 60, 70, 80, 200],
    labels=['Under 50', '50-60', '60-70', '70-80', '80+']
)
print("\n Age breakdown:")
print(age_bins.value_counts().sort_index().rename("Accounts").to_string())


[6] Parent Account Holder Age Distribution
  Mean age              : 67.2
  Median age            : 68.0
  Over 60               : 70%
  Over 70               : 46%
  Over 80               : 23%

 Age breakdown:
PartyAge
Under 50    1429
50-60       1607
60-70       2291
70-80       2171
80+         1977


In [9]:
# =============================================================================
# Step 7: High-Priority Wealth Transfer Accounts
# Criteria: has a Child beneficiary AND positive transfer already occurring.
# Uses max CYTDTransfer per account (delta-safe: max = year-end value).
# =============================================================================

transfer_max = (
    last_per_account_year
    .groupby('FinancialAccountId')['CYTDTransfer']
    .max()
    .reset_index()
    .rename(columns={'CYTDTransfer': 'MaxTransfer'})
)

priority = (
    children_per_account
    .rename(columns={'FinancialAccountId': 'AccountId'})
    .assign(FinancialAccountId=lambda df: df['AccountId'])
    .merge(transfer_max, on='FinancialAccountId', how='left')
    .query('MaxTransfer > 0')
    .sort_values('MaxTransfer', ascending=False)
    .reset_index(drop=True)
)

print("\n" + "=" * 60)
print(f"[7] High-Priority Accounts (Child beneficiary + active transfer): {len(priority):,}")
print("=" * 60)
print("\n  Top 10 by Transfer Amount:")
print(priority.head(10).to_string(index=False))

priority.to_csv('priority_wealth_transfer_accounts_v2.csv', index=False)
print("\n  Saved: priority_wealth_transfer_accounts_v2.csv")


[7] High-Priority Accounts (Child beneficiary + active transfer): 118

  Top 10 by Transfer Amount:
 AccountId  NumChildren  FinancialAccountId  MaxTransfer
  17380029            1            17380029   2657048.83
  18295351            1            18295351   1874477.72
  16720541            1            16720541   1386416.26
  17790164            1            17790164   1375796.73
  15461071            1            15461071   1363054.61
  15940479            1            15940479   1326035.64
  18540760            1            18540760   1261317.10
  17240361            1            17240361   1204600.29
  15960762            1            15960762   1051514.53
  13531258            1            13531258   1050234.09

  Saved: priority_wealth_transfer_accounts_v2.csv
